In [0]:
from pyspark.sql.functions import col,when,timestamp_diff

In [0]:
df = spark.read.table("yuli_demo_nyctaxi.01_bronze.yellow_trips_raw")

In [0]:
#display(df)

In [0]:
from pyspark.sql.functions import max, min
df.agg(max("tpep_pickup_datetime"),min("tpep_pickup_datetime")).display()
# min pickup datetime 2001-01-01T09:23:58.000 in the record (displayed in the result below) -- data cleasing needed

max(tpep_pickup_datetime),min(tpep_pickup_datetime)
2026-06-30T23:59:59.000,2001-01-01T09:23:58.000


##### Filtering out rows with pickup datetime that does not match the timeframe (2026-01 to 2026-06)

In [0]:
df = df.filter("tpep_pickup_datetime >= '2026-01-01' AND tpep_pickup_datetime < '2026-07-01'")

##### Verify once again that all data are within the scope (2026-01 to 2026-08)

In [0]:
df.agg(max("tpep_pickup_datetime"),min("tpep_pickup_datetime")).display()

max(tpep_pickup_datetime),min(tpep_pickup_datetime)
2026-06-30T23:59:59.000,2026-01-01T00:00:00.000


In [0]:
df = df.select(
    when(col("VendorID") == 1, "Creative Mobile Technologies, LLC")
    .when(col("VendorID") == 2, "Curb Mobility, LLC")
    .when(col("VendorID") == 6, "Myle Technologies Inc")
    .when(col("VendorID") == 7, "Helix")
    .otherwise("Unknown")
    .alias("vendor"),
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    timestamp_diff("MINUTE", df.tpep_pickup_datetime, df.tpep_dropoff_datetime).alias("trip_duration"),
    "passenger_count",
    "trip_distance",
    when(col("RatecodeID") == 1, "Standard Rate")
    .when(col("RatecodeID") == 2, "JFK")
    .when(col("RatecodeID") == 3, "Newark")
    .when(col("RatecodeID") == 4, "Nassau or Westchester")
    .when(col("RatecodeID") == 5, "Negotiated Fare")
    .when(col("RatecodeID") == 6, "Group Ride")
    .otherwise("Unknown")
    .alias("rate_type"),
    "store_and_fwd_flag",
    col("PULocationID").alias("pu_location_id"),
    col("DOLocationID").alias("do_location_id"),
    when(col("payment_type") == 0, "Flex Fare trip")
    .when(col("payment_type") == 1, "Credit card")
    .when(col("payment_type") == 2, "Cash")
    .when(col("payment_type") == 3, "No charge")
    .when(col("payment_type") == 4, "Dispute")
    .when(col("payment_type") == 6, "Voided trip")
    .otherwise("Unknown")
    .alias("payment_type"),
    
    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    col("Airport_fee").alias("airport_fee"),
    col("cbd_congestion_fee"),
    "processed_timestamp"
)

In [0]:
#display(df)

In [0]:
df.write.mode("overwrite").saveAsTable("yuli_demo_nyctaxi.02_silver.yellow_trips_cleansed")

In [0]:
spark.read.table("yuli_demo_nyctaxi.02_silver.yellow_trips_cleansed").display()

vendor,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration,passenger_count,trip_distance,rate_type,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,processed_timestamp
Helix,2026-02-01T00:05:57.000,2026-02-01T00:05:57.000,0,1,0.94,Standard Rate,N,107,170,Credit card,7.2,0.0,0.5,0.0,1.0,12.95,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
Helix,2026-02-01T00:35:58.000,2026-02-01T00:35:58.000,0,1,1.93,Standard Rate,N,234,141,Credit card,11.4,0.0,0.5,0.0,1.0,20.58,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
"Curb Mobility, LLC",2026-02-01T00:08:41.000,2026-02-01T00:39:32.000,30,1,9.99,Standard Rate,N,138,68,Credit card,44.3,6.0,0.5,0.0,1.0,67.81,2.5,1.75,0.75,2026-10-08T12:37:13.284Z
"Creative Mobile Technologies, LLC",2026-02-01T00:29:06.000,2026-02-01T00:41:04.000,11,0,1.7,Standard Rate,N,209,13,Credit card,12.8,4.25,0.5,0.0,1.0,22.25,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
"Creative Mobile Technologies, LLC",2026-02-01T00:53:52.000,2026-02-01T01:11:21.000,17,0,3.7,Standard Rate,N,249,229,Credit card,19.8,4.25,0.5,0.0,1.0,31.9,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
"Curb Mobility, LLC",2026-02-01T00:24:29.000,2026-02-01T00:36:01.000,11,1,1.6,Standard Rate,N,113,90,Credit card,12.1,1.0,0.5,0.0,1.0,17.94,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
"Curb Mobility, LLC",2026-02-01T00:40:20.000,2026-02-01T00:54:57.000,14,2,1.73,Standard Rate,N,234,144,Credit card,14.2,1.0,0.5,0.0,1.0,20.95,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
"Curb Mobility, LLC",2026-02-01T00:11:48.000,2026-02-01T00:22:41.000,10,1,2.27,Standard Rate,N,237,151,Credit card,12.8,1.0,0.5,0.0,1.0,19.8,2.5,0.0,0.0,2026-10-08T12:37:13.284Z
"Curb Mobility, LLC",2026-02-01T00:02:38.000,2026-02-01T00:26:38.000,24,1,4.92,Standard Rate,N,148,263,Credit card,26.1,1.0,0.5,0.0,1.0,36.63,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
"Curb Mobility, LLC",2026-02-01T00:05:56.000,2026-02-01T00:22:06.000,16,1,1.98,Standard Rate,N,79,170,Credit card,15.6,1.0,0.5,0.0,1.0,25.62,2.5,0.0,0.75,2026-10-08T12:37:13.284Z
